# 0 - Reset: Limpeza do Ambiente (MinIO)

Este notebook é um utilitário para **resetar completamente** o ambiente de armazenamento de objetos. Ele atua como o passo zero para garantir que os testes sempre comecem com o ambiente limpo.

**O que este script faz:**
1. Conecta-se ao MinIO.
2. Varre os buckets `landing-zone` e `bronze`.
3. Deleta todos os arquivos residuais (CSVs, arquivos Parquet e logs do Delta Lake).
4. Destrói e recria os buckets vazios.

**Por que isso é importante?**
Em arquiteturas de Data Lake, executar rotinas de ingestão repetidas vezes na mesma pasta pode gerar duplicação de dados ou corromper os testes. Rodar este script garante um **início do zero absoluto** para o pipeline de dados.

**Pré-requisitos:** 
- Docker Compose rodando (`docker compose up -d`)
- Arquivo `.env` configurado com as credenciais do MinIO

In [1]:
import os
import boto3
from botocore.client import Config
from dotenv import load_dotenv

# 1. Carrega as credenciais do arquivo .env
load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

print(f"Conectando ao MinIO em: {MINIO_ENDPOINT}...")

# 2. Configura a conexão com o MinIO
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

buckets_para_resetar = [LANDING_BUCKET, BRONZE_BUCKET]

print("Iniciando o reset dos buckets no MinIO...\n")

# 3. Executa a limpeza e recriação
for bucket in buckets_para_resetar:
    try:
        # Verifica se o bucket já existe
        s3_client.head_bucket(Bucket=bucket)
        
        # Lista todos os arquivos (objetos) dentro do bucket
        objetos = s3_client.list_objects_v2(Bucket=bucket)
        
        # Apaga os arquivos um por um (obrigatório antes de apagar o bucket)
        if 'Contents' in objetos:
            print(f"🗑️ Limpando {len(objetos['Contents'])} arquivos do bucket '{bucket}'...")
            for obj in objetos['Contents']:
                s3_client.delete_object(Bucket=bucket, Key=obj['Key'])
        
        # Apaga o bucket vazio
        s3_client.delete_bucket(Bucket=bucket)
        print(f"Bucket '{bucket}' apagado com sucesso.")
        
    except Exception as e:
        # Se o bucket não existir, ele simplesmente cai aqui e segue para a criação
        print(f"ℹBucket '{bucket}' não existia ou estava inacessível.")


print("Processo de reset concluído! Pode iniciar o seu pipeline do zero.")

Conectando ao MinIO em: http://localhost:9020...
Iniciando o reset dos buckets no MinIO...

ℹBucket 'landing-zone' não existia ou estava inacessível.
ℹBucket 'bronze' não existia ou estava inacessível.
Processo de reset concluído! Pode iniciar o seu pipeline do zero.
